# Setup: Quality Metadata Registry

This notebook initializes the central configuration table for the Metadata-Driven Quality Control framework.

In [0]:
from pyspark.sql.types import StructType, StructField, StringType

# 1. Define Schema for the Rules Registry
schema = StructType([
    StructField("table_name", StringType(), False),
    StructField("column_name", StringType(), False),
    StructField("rule_name", StringType(), False),
    StructField("condition", StringType(), False), # SQL condition that MUST BE TRUE (e.g. 'col IS NOT NULL')
    StructField("action", StringType(), False)     # 'QUARANTINE' or 'FAIL'
])

# 2. Initial Rules for Gold Dimensions
initial_rules = [
    # dim_customers rules
    ("dim_customers", "customer_id", "NotNull_ID", "customer_id IS NOT NULL", "QUARANTINE"),
    ("dim_customers", "first_name", "NotNull_FName", "first_name IS NOT NULL", "QUARANTINE"),
    ("dim_customers", "gender", "Valid_Gender", "gender IN ('Male', 'Female', 'n/a')", "QUARANTINE"),
    
    # fact_sales rules
    ("fact_sales", "sales_id", "NotNull_ID", "sales_id IS NOT NULL", "QUARANTINE"),
    ("fact_sales", "sales_amount", "Positive_Sales", "sales_amount >= 0", "QUARANTINE")
]

df_rules = spark.createDataFrame(initial_rules, schema)

# 3. Save to a config/metadata table
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.config")
df_rules.write.format("delta").mode("overwrite").saveAsTable("workspace.config.quality_rules")

print("✓ Quality Metadata Registry initialized at workspace.config.quality_rules")
display(spark.table("workspace.config.quality_rules"))